**main**

In [13]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob

# -----------------------------
# Paths
# -----------------------------
input_dir = r"F:\ALL Works\wiley q1 corrosioin\data\images\main_raw_figs"
output_root = r"F:\ALL Works\wiley q1 corrosioin\figure\updated_new\image_analysis"

image_paths = glob(os.path.join(input_dir, "*.*"))

# -----------------------------
# Unified save function
# -----------------------------
def save_img(img, save_path, is_color=False):
    h, w = img.shape[:2]
    dpi = 300
    figsize = (w / dpi, h / dpi)

    plt.figure(figsize=figsize, dpi=dpi)

    if is_color:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap='gray')

    plt.axis('off')
    plt.savefig(save_path, dpi=dpi, bbox_inches='tight', pad_inches=0)
    plt.close()


# -----------------------------
# Process each image
# -----------------------------
for img_path in image_paths:

    img_name = os.path.splitext(os.path.basename(img_path))[0]
    save_dir = os.path.join(output_root, img_name)
    os.makedirs(save_dir, exist_ok=True)

    print(f"Processing: {img_name}")

    # Load original color image
    image_color = cv2.imread(img_path)

    if image_color is None:
        print(f"Skipping: {img_path}")
        continue

    # Save ORIGINAL (same resolution style as plots)
    save_img(
        image_color,
        os.path.join(save_dir, "Original.svg"),
        is_color=True
    )

    # Convert to grayscale for processing
    image = cv2.cvtColor(image_color, cv2.COLOR_BGR2GRAY)

    # -----------------------------
    # Processing pipeline
    # -----------------------------

    save_img(image, os.path.join(save_dir, "Grayscale.svg"))

    blurred_image = cv2.GaussianBlur(image, (5, 5), 0)
    save_img(blurred_image, os.path.join(save_dir, "Blurred.svg"))

    sobel_x = cv2.Sobel(blurred_image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred_image, cv2.CV_64F, 0, 1, ksize=3)

    sobel_x_abs = cv2.convertScaleAbs(sobel_x)
    sobel_y_abs = cv2.convertScaleAbs(sobel_y)

    save_img(sobel_x_abs, os.path.join(save_dir, "Sobel_X.svg"))
    save_img(sobel_y_abs, os.path.join(save_dir, "Sobel_Y.svg"))

    sobel_combined = cv2.addWeighted(sobel_x_abs, 0.5, sobel_y_abs, 0.5, 0)
    sobel_color = cv2.applyColorMap(sobel_combined, cv2.COLORMAP_JET)
    save_img(sobel_color, os.path.join(save_dir, "Color_Sobel.svg"), is_color=True)

    gradient_magnitude = cv2.magnitude(sobel_x, sobel_y)
    gradient_magnitude = cv2.convertScaleAbs(gradient_magnitude)
    save_img(gradient_magnitude, os.path.join(save_dir, "Gradient_Magnitude.svg"))

    gradient_heatmap = cv2.applyColorMap(gradient_magnitude, cv2.COLORMAP_JET)
    save_img(gradient_heatmap, os.path.join(save_dir, "Gradient_Heatmap.svg"), is_color=True)

    # -----------------------------
    # Roughness metric
    # -----------------------------
    roughness = np.std(gradient_magnitude)

    with open(os.path.join(save_dir, "metrics.txt"), "w") as f:
        f.write(f"Surface Roughness (Std Dev): {roughness:.4f}")

    # -----------------------------
    # Summary figure (publication style)
    # -----------------------------
    h, w = image.shape[:2]
    dpi = 300
    figsize = (w / dpi, h / dpi)

    plt.figure(figsize=(15, 10), dpi=dpi)

    plt.subplot(2, 3, 1)
    plt.imshow(cv2.cvtColor(image_color, cv2.COLOR_BGR2RGB))
    plt.title('Original')
    plt.axis('off')

    plt.subplot(2, 3, 2)
    plt.imshow(image, cmap='gray')
    plt.title('Grayscale')
    plt.axis('off')

    plt.subplot(2, 3, 3)
    plt.imshow(sobel_x_abs, cmap='gray')
    plt.title('Sobel X')
    plt.axis('off')

    plt.subplot(2, 3, 4)
    plt.imshow(sobel_y_abs, cmap='gray')
    plt.title('Sobel Y')
    plt.axis('off')

    plt.subplot(2, 3, 5)
    plt.imshow(cv2.cvtColor(sobel_color, cv2.COLOR_BGR2RGB))
    plt.title('Color Sobel')
    plt.axis('off')

    plt.subplot(2, 3, 6)
    plt.imshow(gradient_magnitude, cmap='jet')
    plt.title('Gradient Magnitude')
    plt.colorbar()

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "summary.svg"), dpi=300, bbox_inches='tight')
    plt.close()

print("✅ All images processed successfully.")

Processing: 6325648265168228345_121
Processing: 6325648265168228348_121
Processing: 6325648265168228353_121
Processing: 6327695457034899634_121
Processing: 6327695457034899635_121
Processing: 6327695457034899637_121
Processing: 6327695457034899638_121
Processing: 6327695457034899639_121
Processing: 6327695457034899640_121
Processing: 6327695457034899641_121
Processing: 6327695457034899664_121
Processing: 6327695457034899665_121
Processing: 6327695457034899687_121
✅ All images processed successfully.
